In [ ]:
from fastapi import (
    APIRouter,
    HTTPException,
    status,
    Depends,
    Query,
)

from app.api.auth import get_current_user


router = APIRouter(
    prefix="/analytics",
    tags=["Analytics"],
)


def get_database():
    from app.main import app

    database = getattr(app.state, "database", None)

    if database is None:
        raise RuntimeError("Database is not initialized.")

    return database


def verify_project_ownership(
    database,
    project_id: str,
    user_id: str,
):
    project = database.collection("projects").find_one(
        {
            "id": project_id,
            "user_id": user_id,
        }
    )

    if project is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="Project not found.",
        )

    return project


def clean(document):
    result = dict(document)
    result.pop("_id", None)
    return result


def assessment_metrics(documents):
    completed = [
        item
        for item in documents
        if item.get("status") == "completed"
    ]

    percentages = []
    for item in completed:
        score = item.get("score")
        maximum = item.get("max_score")
        if score is not None and maximum is not None and float(maximum) > 0:
            percentages.append(
                max(0.0, min(1.0, float(score) / float(maximum)))
            )

    return {
        "total": len(documents),
        "completed": len(completed),
        "average_score": round(sum(percentages) / len(percentages), 4) if percentages else 0.0,
        "questions_answered": sum(len(item.get("answers", [])) for item in completed),
    }


def mastery_metrics(documents):
    scores = [float(item.get("score", 0.0)) for item in documents]
    trend_counts = {
        "improving": 0,
        "stable": 0,
        "needs_attention": 0,
    }

    for item in documents:
        trend = item.get("trend", "stable")
        if trend in trend_counts:
            trend_counts[trend] += 1

    return {
        "concept_count": len(documents),
        "average_score": round(sum(scores) / len(scores), 4) if scores else 0.0,
        "trend_counts": trend_counts,
    }


def activity_metrics(documents):
    event_counts = {}
    for item in documents:
        event_type = str(item.get("event_type", "unknown"))
        event_counts[event_type] = event_counts.get(event_type, 0) + 1

    return {
        "event_count": len(documents),
        "event_counts": event_counts,
    }


def ai_metrics(database, query):
    documents = list(
        database.collection("ai_usage").find(
            query,
            {
                "_id": 0,
                "success": 1,
                "total_tokens": 1,
                "latency_ms": 1,
            },
        )
    )

    latencies = [
        float(item["latency_ms"])
        for item in documents
        if item.get("latency_ms") is not None
    ]

    return {
        "request_count": len(documents),
        "successful_requests": sum(1 for item in documents if item.get("success") is True),
        "failed_requests": sum(1 for item in documents if item.get("success") is False),
        "total_tokens": sum(int(item.get("total_tokens") or 0) for item in documents),
        "average_latency_ms": round(sum(latencies) / len(latencies), 2) if latencies else 0.0,
    }


@router.get("/project/{project_id}")
async def get_project_analytics(
    project_id: str,
    current_user=Depends(get_current_user),
):
    """Return learning, mastery, activity and AI analytics for one project."""

    database = get_database()
    verify_project_ownership(database, project_id, current_user.id)

    scope = {"project_id": project_id, "user_id": current_user.id}

    activities = list(database.collection("activities").find(scope, {"_id": 0}))
    assessments = list(database.collection("assessments").find(scope, {"_id": 0, "status": 1, "score": 1, "max_score": 1, "answers": 1}))
    mastery = list(database.collection("mastery").find(scope, {"_id": 0, "score": 1, "trend": 1}))

    return {
        "project_id": project_id,
        "activity": activity_metrics(activities),
        "material_count": database.collection("materials").count_documents(scope),
        "conversation_count": database.collection("conversations").count_documents(scope),
        "assessments": assessment_metrics(assessments),
        "mastery": mastery_metrics(mastery),
        "ai": ai_metrics(database, scope),
    }


@router.get("/global")
async def get_global_analytics(
    current_user=Depends(get_current_user),
):
    """Return analytics aggregated across all projects owned by the user."""

    database = get_database()
    user_scope = {"user_id": current_user.id}

    activities = list(database.collection("activities").find(user_scope, {"_id": 0}))
    assessments = list(database.collection("assessments").find(user_scope, {"_id": 0, "status": 1, "score": 1, "max_score": 1, "answers": 1}))
    mastery = list(database.collection("mastery").find(user_scope, {"_id": 0, "score": 1, "trend": 1}))

    return {
        "project_count": database.collection("projects").count_documents(user_scope),
        "space_count": database.collection("spaces").count_documents(user_scope),
        "material_count": database.collection("materials").count_documents(user_scope),
        "conversation_count": database.collection("conversations").count_documents(user_scope),
        "activity": activity_metrics(activities),
        "assessments": assessment_metrics(assessments),
        "mastery": mastery_metrics(mastery),
        "ai": ai_metrics(database, user_scope),
    }


@router.get("/activity")
async def get_activity(
    limit: int = Query(default=50, ge=1, le=200),
    current_user=Depends(get_current_user),
):
    """Return recent activity for the authenticated user."""

    database = get_database()

    documents = database.collection("activities").find(
        {"user_id": current_user.id}
    ).sort("created_at", -1).limit(limit)

    return [clean(document) for document in documents]


@router.get("/activity/project/{project_id}")
async def get_project_activity(
    project_id: str,
    limit: int = Query(default=50, ge=1, le=200),
    current_user=Depends(get_current_user),
):
    """Return recent activity for an authorized project."""

    database = get_database()
    verify_project_ownership(database, project_id, current_user.id)

    documents = database.collection("activities").find(
        {"project_id": project_id, "user_id": current_user.id}
    ).sort("created_at", -1).limit(limit)

    return [clean(document) for document in documents]
